# Notebook 03 — Feature Engineering

**Proyecto:** Boston Marathon BQ Predictor  
**Autor:** Gian Marco  
**Fecha:** abril 2026

## Objetivos

1. Limpiar columnas no útiles o con leakage
2. Hacer encoding de variables categóricas (Country, Race, Race Category)
3. Crear features derivadas (Is_Home_Country, Age_Squared)
4. Target encoding con smoothing y K-Fold para Race (evitando leakage)
5. Guardar datasets finales listos para modelar
6. Exportar pipeline reutilizable a `src/preprocessing.py`

## Decisiones aplicadas

Todas documentadas en `DECISIONS.md`. Resumen:
- `Country`: agrupar baja frecuencia (<500 corredores) como "Other"
- `Race`: target encoding K-Fold para evitar leakage
- Features eliminadas: todas las que causen leakage directo o indirecto

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold
import joblib
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROCESSED_DATA_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print('Setup listo')

Setup listo


---
## 1. Cargar datos

In [2]:
TRAIN_DIR = Path('../data/train')
TEST_DIR = Path('../data/test')

train = pd.read_csv(TRAIN_DIR / 'train.csv')
test = pd.read_csv(TEST_DIR / 'test.csv')

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')
print(f'\nColumnas actuales: {train.columns.tolist()}')

Train: (225356, 15)
Test:  (74644, 15)

Columnas actuales: ['Year', 'Race', 'Country', 'Gender', 'Age', 'Finish', 'Overall Place', 'Gender Place', 'Race_City', 'Race_State', 'Finishers', 'Category', 'Age Bracket', 'Standard', 'es_BQ']


---
## 2. Eliminar columnas con leakage o redundantes

Estas columnas NO pueden usarse como features:

- **`Finish`**: es el tiempo final del corredor, usado para calcular el target. Leakage directo.
- **`Standard`**: es el tiempo BQ por categoría edad/género, usado para calcular el target. Leakage directo.
- **`Overall Place`** y **`Gender Place`**: son función directa del tiempo final. Leakage indirecto.
- **`Age Bracket`**: redundante con `Age` (es su discretización).
- **`Race_City`** y **`Race_State`**: redundantes con `Race`.

In [3]:
COLS_TO_DROP = [
    'Finish',           # Leakage: usado para construir target
    'Standard',         # Leakage: usado para construir target
    'Overall Place',    # Leakage: función del tiempo
    'Gender Place',     # Leakage: función del tiempo
    'Age Bracket',      # Redundante con Age
    'Race_City',        # Redundante con Race
    'Race_State',       # Redundante con Race
]

# Si existe Finish_h de EDA, también eliminar
extra_cols = [c for c in ['Finish_h'] if c in train.columns]

train = train.drop(columns=COLS_TO_DROP + extra_cols, errors='ignore')
test = test.drop(columns=COLS_TO_DROP + extra_cols, errors='ignore')

print(f'Train tras drop: {train.shape}')
print(f'Columnas restantes: {train.columns.tolist()}')

Train tras drop: (225356, 8)
Columnas restantes: ['Year', 'Race', 'Country', 'Gender', 'Age', 'Finishers', 'Category', 'es_BQ']


---
## 3. Agrupar países de baja frecuencia

Hay más de 200 países en el dataset, pero muchos tienen muy pocos corredores. Un país con 3 corredores no aporta señal estadística fiable al modelo y genera ruido.

Agrupamos en "Other" los países con menos de 500 corredores en train.

**Importante:** el umbral se calcula SOLO sobre train, y se aplica igual a test para evitar leakage.

In [4]:
MIN_COUNTRY_COUNT = 500

# Calcular frecuencias en TRAIN (nunca usar test para decisiones)
country_counts = train['Country'].value_counts()
countries_to_keep = country_counts[country_counts >= MIN_COUNTRY_COUNT].index.tolist()

print(f'Países originales: {train["Country"].nunique()}')
print(f'Países con mas de {MIN_COUNTRY_COUNT} corredores: {len(countries_to_keep)}')
print(f'\nPaíses que se mantienen: {sorted(countries_to_keep)}')

# Aplicar a train y test
train['Country_grouped'] = train['Country'].where(train['Country'].isin(countries_to_keep), 'Other')
test['Country_grouped'] = test['Country'].where(test['Country'].isin(countries_to_keep), 'Other')

print(f'\nPaíses finales en train: {train["Country_grouped"].nunique()}')
print(f'% corredores en Other: {(train["Country_grouped"]=="Other").mean()*100:.2f}%')

Países originales: 163
Países con mas de 500 corredores: 15

Países que se mantienen: ['AU', 'BR', 'CA', 'DE', 'ES', 'FR', 'GB', 'IE', 'IT', 'JP', 'MX', 'NL', 'PL', 'TH', 'US']

Países finales en train: 16
% corredores en Other: 4.07%


---
## 4. Feature derivada: `Is_Home_Country`

Un corredor que compite en su propio país tiene ventajas logísticas (menos jet lag, conocimiento del circuito, clima familiar). Es una feature binaria simple pero potencialmente útil.

Comparamos el `Country` del corredor con el país en el que se celebra la carrera. Esto requiere inferir el país de cada carrera desde `Race_State` o el nombre de la carrera (Races.csv).

Estrategia simple: las carreras con `Race_State` no-nulo son de US, el resto las mapeamos por nombre.

In [5]:
# Recargar Races.csv para obtener el país de cada carrera
races_meta = pd.read_csv(PROCESSED_DATA_DIR.parent / 'raw' / 'Races.csv')

# Mapping manual de race a country code (carreras principales no-US)
race_country_map = {
    'London Marathon': 'GB',
    'Berlin Marathon': 'DE',
    'Tokyo Marathon': 'JP',
    'RnR Madrid Marathon': 'ES',
    'Zurich Marato Barcelona': 'ES',
    'Zurich Maraton Sevilla': 'ES',
    'Zurich Marathon': 'CH',
    'Mexico City Marathon': 'MX',
    'Marathon Pour Tous': 'FR',
    'Athens Marathon': 'GR',
    'Amsterdam Marathon': 'NL',
    'Rotterdam Marathon': 'NL',
    'Valencia Marathon': 'ES',
    'Paris Marathon': 'FR',
    'Copenhagen Marathon': 'DK',
    'Dublin Marathon': 'IE',
    'Stockholm Marathon': 'SE',
    'Vienna City Marathon': 'AT',
    'Cape Town Marathon': 'ZA',
    'Comrades Marathon': 'ZA',
    'Sydney Marathon': 'AU',
    'Melbourne Marathon': 'AU',
    'Toronto Waterfront Marathon': 'CA',
    'Ottawa Marathon': 'CA',
    'LANZAROTE INTERNATIONAL MARATHON': 'ES',
}

def get_race_country(race_name):
    """Asigna el país donde se celebra la carrera. Default: US."""
    return race_country_map.get(race_name, 'US')

train['Race_Country'] = train['Race'].apply(get_race_country)
test['Race_Country'] = test['Race'].apply(get_race_country)

# Crear Is_Home_Country
train['Is_Home_Country'] = (train['Country'] == train['Race_Country']).astype(int)
test['Is_Home_Country'] = (test['Country'] == test['Race_Country']).astype(int)

print(f'Carreras mapeadas no-US: {len(race_country_map)}')
print(f'Resto asumidas US (mayoría del dataset)')
print(f'\n% corredores home country en train: {train["Is_Home_Country"].mean()*100:.2f}%')
print(f'\n% BQ en home vs away (train):')
print(train.groupby('Is_Home_Country')['es_BQ'].agg(['mean', 'count']).round(4))

Carreras mapeadas no-US: 25
Resto asumidas US (mayoría del dataset)

% corredores home country en train: 82.79%

% BQ en home vs away (train):
                   mean   count
Is_Home_Country                
0                0.1875   38777
1                0.1235  186579


---
## 5. Encoding de `Race_Category`

La columna `Category` de Races.csv tiene 4 valores: Minor, Moderate, Steep, Very Steep (más NaN). Es ordinal: captura la dificultad del circuito.

Encoding manual (ordinal): Minor=0, Moderate=1, Steep=2, Very Steep=3. Los NaN los llenamos con la moda del train.

In [6]:
category_mapping = {
    'Minor': 0,
    'Moderate': 1,
    'Steep': 2,
    'Very Steep': 3,
}

train['Race_Category_enc'] = train['Category'].map(category_mapping)
test['Race_Category_enc'] = test['Category'].map(category_mapping)

# Imputar NaN con la moda del train
mode_category = train['Race_Category_enc'].mode()[0]
train['Race_Category_enc'] = train['Race_Category_enc'].fillna(mode_category)
test['Race_Category_enc'] = test['Race_Category_enc'].fillna(mode_category)

print(f'Valor usado para imputar NaN: {mode_category}')
print(f'\nDistribución en train:')
print(train['Race_Category_enc'].value_counts().sort_index())

Valor usado para imputar NaN: 0.0

Distribución en train:
Race_Category_enc
0.0    216765
1.0      2490
2.0      3747
3.0      2354
Name: count, dtype: int64


---
## 6. Target encoding con K-Fold para `Race`

**El problema:** la variable `Race` tiene ~500 valores únicos. One-hot generaría 500 columnas (ineficiente). Target encoding simple (reemplazar cada carrera por su % BQ medio) causaría **leakage**: estarías usando el target para crear una feature.

**La solución — K-Fold Target Encoding con Smoothing:**

1. Dividimos el train en K folds
2. Para cada fold, calculamos el % BQ medio de cada carrera usando SOLO los otros K-1 folds
3. Aplicamos ese encoding al fold actual
4. Para el test, usamos el encoding calculado sobre TODO el train

**Smoothing:** carreras con pocos corredores en un fold tendrían estadísticas ruidosas. Aplicamos un promedio ponderado con la media global según el número de samples.

Fórmula: `encoding(race) = (n * mean_race + m * mean_global) / (n + m)`, donde `n` = conteo de esa carrera, `m` = factor de suavizado.

In [7]:
def kfold_target_encode(train_df, test_df, col, target, n_splits=5, smoothing=10, random_state=42):
    """
    Target encoding con K-Fold para evitar leakage.
    
    Parameters
    ----------
    train_df : DataFrame con col y target
    test_df  : DataFrame con col (sin target necesariamente)
    col      : nombre de la columna categórica
    target   : nombre de la columna target
    n_splits : folds para el encoding
    smoothing: factor de suavizado (bayesiano simple)
    """
    global_mean = train_df[target].mean()
    
    # --- Encoding del train (K-Fold) ---
    train_encoded = np.zeros(len(train_df))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(train_df)):
        fold_train = train_df.iloc[train_idx]
        
        # Stats por categoría en el fold
        agg = fold_train.groupby(col)[target].agg(['mean', 'count'])
        
        # Smoothing
        smoothed = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
        
        # Aplicar al fold de validación
        val_categories = train_df.iloc[val_idx][col]
        train_encoded[val_idx] = val_categories.map(smoothed).fillna(global_mean).values
    
    # --- Encoding del test (usando TODO el train) ---
    agg_full = train_df.groupby(col)[target].agg(['mean', 'count'])
    smoothed_full = (agg_full['count'] * agg_full['mean'] + smoothing * global_mean) / (agg_full['count'] + smoothing)
    test_encoded = test_df[col].map(smoothed_full).fillna(global_mean).values
    
    return train_encoded, test_encoded, smoothed_full


# Aplicar a Race
race_enc_train, race_enc_test, race_encoding_map = kfold_target_encode(
    train, test, col='Race', target='es_BQ', n_splits=5, smoothing=10
)

train['Race_te'] = race_enc_train
test['Race_te'] = race_enc_test

print('Encoding aplicado a Race')
print(f'\nRange del encoding en train: [{train["Race_te"].min():.4f}, {train["Race_te"].max():.4f}]')
print(f'Media del encoding en train: {train["Race_te"].mean():.4f} (debería ser ~{train["es_BQ"].mean():.4f})')

# Top 10 carreras con mayor encoding (más "BQ-friendly")
print('\nTop 10 carreras con mayor Race_te (encoding completo con todo el train):')
print(race_encoding_map.sort_values(ascending=False).head(10).round(4))

Encoding aplicado a Race

Range del encoding en train: [0.0045, 0.8543]
Media del encoding en train: 0.1348 (debería ser ~0.1345)

Top 10 carreras con mayor Race_te (encoding completo con todo el train):
Race
AbbottWMM Wanda Age Group World Championship    0.8510
Boston Marathon                                 0.5022
Last Chance BQ Chicagoland Marathon             0.4552
Last Chance BQ Grand Rapids                     0.4465
The Light At The End of the Tunnel              0.3998
REVEL Mt. Charleston                            0.3814
REVEL Big Bear                                  0.3716
Mountains 2 Beach Marathon                      0.3280
Tunnel Lite Marathon                            0.3239
California International Marathon               0.3169
dtype: float64


---
## 7. Feature derivada: `Age_Squared`

La relación entre edad y probabilidad de BQ NO es lineal: vimos en el EDA que los 60-69 años tienen tasas más altas que los 30-40. Un modelo lineal no capturaría esto con solo `Age`.

Añadir `Age_Squared` permite a modelos lineales capturar esta curvatura. Para modelos basados en árboles (Random Forest, XGBoost) esto es redundante, pero no hace daño.

In [8]:
train['Age_Squared'] = train['Age'] ** 2
test['Age_Squared'] = test['Age'] ** 2

print(f'Age range: [{train["Age"].min()}, {train["Age"].max()}]')
print(f'Age_Squared range: [{train["Age_Squared"].min()}, {train["Age_Squared"].max()}]')

Age range: [18.0, 85.0]
Age_Squared range: [324.0, 7225.0]


---
## 8. One-Hot Encoding de variables categóricas finales

Solo queda encoding simple de variables de baja cardinalidad:

- `Gender` (M, F) → binaria (Male=1, Female=0)
- `Country_grouped` (~25-30 valores) → one-hot

In [9]:
# Gender a binaria
train['Gender_M'] = (train['Gender'] == 'M').astype(int)
test['Gender_M'] = (test['Gender'] == 'M').astype(int)

# Country one-hot (asegurando que train y test tengan las mismas columnas)
train_dummies = pd.get_dummies(train['Country_grouped'], prefix='Country', dtype=int)
test_dummies = pd.get_dummies(test['Country_grouped'], prefix='Country', dtype=int)

# Alinear columnas (test puede tener menos categorías)
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

train = pd.concat([train, train_dummies], axis=1)
test = pd.concat([test, test_dummies], axis=1)

print(f'Columnas one-hot de Country creadas: {len(train_dummies.columns)}')
print(f'Muestra: {train_dummies.columns.tolist()[:5]}...')

Columnas one-hot de Country creadas: 16
Muestra: ['Country_AU', 'Country_BR', 'Country_CA', 'Country_DE', 'Country_ES']...


---
## 9. Selección final de features y guardado

Eliminamos las columnas originales categóricas que ya están encodeadas, y dejamos solo las features finales para el modelo.

In [10]:
# Columnas originales a eliminar (ya están encodeadas)
COLS_ENCODED_DROP = [
    'Race',             # encodeada en Race_te
    'Country',          # encodeada en Country_grouped y one-hot
    'Country_grouped',  # intermedia, one-hot la reemplaza
    'Gender',           # encodeada en Gender_M
    'Category',         # encodeada en Race_Category_enc
    'Race_Country',     # solo usada para Is_Home_Country
    'Finishers',        # info de Races.csv, redundante con Race_te
]

train_final = train.drop(columns=COLS_ENCODED_DROP, errors='ignore')
test_final = test.drop(columns=COLS_ENCODED_DROP, errors='ignore')

# Reordenar para que el target quede al final
target_col = 'es_BQ'
feature_cols = [c for c in train_final.columns if c != target_col]
train_final = train_final[feature_cols + [target_col]]
test_final = test_final[feature_cols + [target_col]]

print(f'Train final: {train_final.shape}')
print(f'Test final:  {test_final.shape}')
print(f'\nFeatures finales ({len(feature_cols)}):')
for c in feature_cols:
    print(f'  - {c} ({train_final[c].dtype})')

# Guardar
train_final.to_csv(TRAIN_DIR / 'train_features.csv', index=False)
test_final.to_csv(TEST_DIR / 'test_features.csv', index=False)

print('\nGuardados:')
print(f'  {TRAIN_DIR / "train_features.csv"}')
print(f'  {TEST_DIR / "test_features.csv"}')

Train final: (225356, 24)
Test final:  (74644, 24)

Features finales (23):
  - Year (int64)
  - Age (float64)
  - Is_Home_Country (int64)
  - Race_Category_enc (float64)
  - Race_te (float64)
  - Age_Squared (float64)
  - Gender_M (int64)
  - Country_AU (int64)
  - Country_BR (int64)
  - Country_CA (int64)
  - Country_DE (int64)
  - Country_ES (int64)
  - Country_FR (int64)
  - Country_GB (int64)
  - Country_IE (int64)
  - Country_IT (int64)
  - Country_JP (int64)
  - Country_MX (int64)
  - Country_NL (int64)
  - Country_Other (int64)
  - Country_PL (int64)
  - Country_TH (int64)
  - Country_US (int64)

Guardados:
  ../data/train/train_features.csv
  ../data/test/test_features.csv


---
## 10. Guardar artefactos para reutilización

El encoding de `Race` (con smoothing, calculado sobre todo el train) debe guardarse para usarlo en predicciones futuras (ej. la app Streamlit).

In [11]:
artifacts = {
    'countries_to_keep': countries_to_keep,
    'race_country_map': race_country_map,
    'category_mapping': category_mapping,
    'race_encoding_map': race_encoding_map,
    'mode_category': mode_category,
    'global_mean_bq': train['es_BQ'].mean(),
    'feature_cols': feature_cols,
}

joblib.dump(artifacts, MODELS_DIR / 'preprocessing_artifacts.joblib')
print(f'Artefactos guardados en {MODELS_DIR / "preprocessing_artifacts.joblib"}')
print(f'\nContenido: {list(artifacts.keys())}')

Artefactos guardados en ../models/preprocessing_artifacts.joblib

Contenido: ['countries_to_keep', 'race_country_map', 'category_mapping', 'race_encoding_map', 'mode_category', 'global_mean_bq', 'feature_cols']


---
## 11. Sanity check final

Verificaciones antes de cerrar el notebook:
- No hay NaN en features
- Tipos de dato correctos
- Dimensiones coherentes
- El target se conserva igual

In [12]:
print('SANITY CHECKS')
print('=' * 60)

# 1. NaN check
nulls_train = train_final.isnull().sum().sum()
nulls_test = test_final.isnull().sum().sum()
print(f'\n1. Nulos en train: {nulls_train}')
print(f'   Nulos en test:  {nulls_test}')
assert nulls_train == 0, 'Hay nulos en train'
assert nulls_test == 0, 'Hay nulos en test'

# 2. Target conservado
print(f'\n2. % BQ train: {train_final["es_BQ"].mean()*100:.2f}%')
print(f'   % BQ test:  {test_final["es_BQ"].mean()*100:.2f}%')

# 3. Dimensiones
print(f'\n3. Train: {train_final.shape[0]:,} filas x {train_final.shape[1]} cols')
print(f'   Test:  {test_final.shape[0]:,} filas x {test_final.shape[1]} cols')
assert train_final.shape[1] == test_final.shape[1], 'Distinto número de columnas'

# 4. Tipos (todos numéricos)
non_numeric = train_final.dtypes[~train_final.dtypes.apply(lambda x: np.issubdtype(x, np.number))]
print(f'\n4. Columnas no numéricas: {len(non_numeric)}')
if len(non_numeric) > 0:
    print(non_numeric)

# 5. Columnas coinciden
assert list(train_final.columns) == list(test_final.columns), 'Columnas distintas'
print(f'\n5. Columnas train == test: OK')

print('\nTodos los checks pasados. Datasets listos para modelar en Notebook 04.')

SANITY CHECKS

1. Nulos en train: 0
   Nulos en test:  0

2. % BQ train: 13.45%
   % BQ test:  14.30%

3. Train: 225,356 filas x 24 cols
   Test:  74,644 filas x 24 cols

4. Columnas no numéricas: 0

5. Columnas train == test: OK

Todos los checks pasados. Datasets listos para modelar en Notebook 04.


---
## Resumen del Notebook 03

### Features finales creadas

| Feature | Tipo | Descripción |
|---|---|---|
| `Age` | Numérica | Edad del corredor |
| `Age_Squared` | Numérica | Captura no-linealidad del efecto edad |
| `Year` | Numérica | Usado en split temporal |
| `Race_Category_enc` | Ordinal | Dificultad del circuito (0=Minor, 3=Very Steep) |
| `Race_te` | Numérica | Target encoding de Race (K-Fold con smoothing) |
| `Is_Home_Country` | Binaria | 1 si el corredor compite en su país |
| `Gender_M` | Binaria | 1 si Masculino |
| `Country_*` | Binarias | One-hot de Country_grouped (~25-30) |

### Decisiones clave tomadas

1. **Sin leakage:** eliminadas todas las features que derivan del tiempo de finish
2. **Country agrupado:** umbral 500, reduce de ~222 a ~25-30 categorías útiles
3. **Race con K-Fold TE:** evita leakage manteniendo la señal predictiva
4. **Age_Squared:** permite a modelos lineales capturar la no-linealidad edad-BQ